In [ ]:
!pip install xgboost shap imbalanced-learn pandasql -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, roc_curve, accuracy_score
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
import shap
import pandasql as ps
import joblib
plt.rcParams["figure.figsize"] = (12, 6)
sns.set_style("whitegrid")
print("All libraries loaded!")


In [ ]:
# Correct URL - IBM official repository
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")
print(f"Churn rate: {(df['Churn']=='Yes').mean()*100:.1f}%")
df.head()

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"].fillna(df["TotalCharges"].median(), inplace=True)
df.drop("customerID", axis=1, inplace=True)
df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"})
df["TenureBucket"] = pd.cut(df["tenure"], bins=[0,12,24,48,72],
    labels=["0-12 months","13-24 months","25-48 months","49-72 months"])
print("Done! Missing:", df.isnull().sum().sum())

In [ ]:
q1 = """SELECT Contract, COUNT(*) AS total,
    ROUND(100.0*SUM(CASE WHEN Churn="Yes" THEN 1 ELSE 0 END)/COUNT(*),1) AS churn_pct
    FROM df GROUP BY Contract ORDER BY churn_pct DESC"""
print("SQL1: Churn by Contract"); print(ps.sqldf(q1, locals()))

q2 = """SELECT Churn, COUNT(*) AS customers,
    ROUND(SUM(MonthlyCharges),2) AS monthly_revenue,
    ROUND(AVG(tenure),1) AS avg_tenure
    FROM df GROUP BY Churn"""
r2 = ps.sqldf(q2, locals())
print("SQL2: Revenue at Risk"); print(r2)
print(f"Revenue at Risk: ${r2[r2['Churn']=='Yes']['monthly_revenue'].values[0]:,.2f}")

q3 = """SELECT PaymentMethod,
    ROUND(100.0*SUM(CASE WHEN Churn="Yes" THEN 1 ELSE 0 END)/COUNT(*),1) AS churn_pct
    FROM df GROUP BY PaymentMethod ORDER BY churn_pct DESC"""
print("SQL3: Churn by Payment"); print(ps.sqldf(q3, locals()))

q4 = """SELECT Contract, Churn, COUNT(*) AS customers,
    ROUND(AVG(tenure),1) AS avg_tenure, ROUND(AVG(MonthlyCharges),2) AS avg_charges
    FROM df GROUP BY Contract, Churn ORDER BY Contract, Churn"""
print("SQL4: Cohort Analysis"); print(ps.sqldf(q4, locals()))

q5 = """SELECT TenureBucket, Contract, COUNT(*) AS total,
    ROUND(100.0*SUM(CASE WHEN Churn="Yes" THEN 1 ELSE 0 END)/COUNT(*),1) AS churn_pct
    FROM df GROUP BY TenureBucket, Contract ORDER BY churn_pct DESC LIMIT 10"""
print("SQL5: Highest Risk Segments"); print(ps.sqldf(q5, locals()))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))
axes[0].pie(df["Churn"].value_counts(), labels=["Retained","Churned"],
    colors=["#2563EB","#DC2626"], autopct="%1.1f%%", startangle=90)
axes[0].set_title("Overall Churn Distribution", fontsize=14, fontweight="bold")
churn_c = df.groupby("Contract")["Churn"].apply(
    lambda x: (x=="Yes").sum()/len(x)*100).reset_index()
churn_c.columns = ["Contract","ChurnRate"]
bars = axes[1].bar(churn_c["Contract"], churn_c["ChurnRate"],
    color=["#DC2626","#F97316","#16A34A"])
axes[1].set_title("Churn Rate by Contract Type", fontweight="bold")
axes[1].set_ylabel("Churn Rate (%)")
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
for bar, val in zip(bars, churn_c["ChurnRate"]):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
        f"{val:.1f}%", ha="center", fontweight="bold")
plt.tight_layout()
plt.savefig("churn_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
#---- CELL 7: EDA Plot 2 - Tenure Analysis ----
fig, axes = plt.subplots(1, 2, figsize=(14,5))
for val, col, lbl in [("No","#2563EB","Retained"),("Yes","#DC2626","Churned")]:
    s = df[df["Churn"]==val]
    axes[0].scatter(s["tenure"], s["MonthlyCharges"], alpha=0.3, color=col, label=lbl, s=15)
axes[0].set_xlabel("Tenure (months)"); axes[0].set_ylabel("Monthly Charges ($)")
axes[0].set_title("Tenure vs Monthly Charges", fontweight="bold"); axes[0].legend()
tc = df.groupby("TenureBucket")["Churn"].apply(
    lambda x: (x=="Yes").sum()/len(x)*100).reset_index()
tc.columns = ["TenureBucket","ChurnRate"]
axes[1].bar(tc["TenureBucket"], tc["ChurnRate"],
    color=["#DC2626","#F97316","#3B82F6","#16A34A"])
axes[1].set_title("Churn Rate by Tenure Bucket", fontweight="bold")
axes[1].set_ylabel("Churn Rate (%)"); axes[1].tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.savefig("tenure_analysis.png", dpi=150, bbox_inches="tight"); plt.show()


In [ ]:
# ---- CELL 8: EDA Plot 3 - Categorical Features ----
features = ["InternetService","PaymentMethod","SeniorCitizen","Partner"]
fig, axes = plt.subplots(2, 2, figsize=(16,10)); axes = axes.flatten()
for i, col in enumerate(features):
    cr = df.groupby(col)["Churn"].apply(
        lambda x: (x=="Yes").sum()/len(x)*100).reset_index()
    cr.columns = [col,"ChurnRate"]
    bars = axes[i].bar(cr[col].astype(str), cr["ChurnRate"],
        color=sns.color_palette("RdYlGn_r", len(cr)))
    axes[i].set_title(f"Churn Rate by {col}", fontweight="bold")
    axes[i].set_ylabel("Churn Rate (%)")
    axes[i].tick_params(axis="x", rotation=15)
    for bar, val in zip(bars, cr["ChurnRate"]):
        axes[i].text(bar.get_x()+bar.get_width()/2,
            bar.get_height()+0.3, f"{val:.1f}%", ha="center", fontsize=10)
plt.suptitle("Churn by Key Segments", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig("categorical_churn.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# ---- CELL 9: Feature Engineering + SMOTE ----
df_model = df.copy()
df_model.drop("TenureBucket", axis=1, inplace=True)
binary_cols = ["Partner","Dependents","PhoneService","PaperlessBilling","Churn","SeniorCitizen"]
for col in binary_cols:
    df_model[col] = (df_model[col]=="Yes").astype(int)
le = LabelEncoder()
for col in df_model.select_dtypes(include="object").columns:
    df_model[col] = le.fit_transform(df_model[col])
X = df_model.drop("Churn", axis=1)
y = df_model["Churn"]
print(f"Before SMOTE: {y.value_counts().to_dict()}")
X_res, y_res = SMOTE(random_state=42).fit_resample(X, y)
print(f"After SMOTE:  {pd.Series(y_res).value_counts().to_dict()}")
X_train, X_test, y_train, y_test = train_test_split(
    X_res, y_res, test_size=0.2, random_state=42, stratify=y_res)
print(f"Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}")

In [ ]:
# ---- CELL 10: Train 3 Models ----
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest":       RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost":             XGBClassifier(n_estimators=100, random_state=42,
                               eval_metric="logloss", verbosity=0)
}
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:,1]
    results[name] = {"model":model, "y_pred":y_pred, "y_prob":y_prob,
        "accuracy": accuracy_score(y_test, y_pred),
        "roc_auc":  roc_auc_score(y_test, y_prob)}
    print(f"\n=== {name} ===")
    print(classification_report(y_test, y_pred, target_names=["Retained","Churned"]))
    print(f"ROC-AUC: {results[name]['roc_auc']:.4f}")


In [ ]:
# ---- CELL 11: ROC Curve + Model Comparison ----
fig, axes = plt.subplots(1, 2, figsize=(15,5))
colors = ["#2563EB","#16A34A","#DC2626"]
for (name, res), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, res["y_prob"])
    axes[0].plot(fpr, tpr, color=color, lw=2, label=f"{name} (AUC={res['roc_auc']:.3f})")
axes[0].plot([0,1],[0,1],"k--")
axes[0].set_xlabel("False Positive Rate"); axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curves", fontweight="bold"); axes[0].legend(loc="lower right")
names = list(results.keys())
accs = [r["accuracy"]*100 for r in results.values()]
aucs = [r["roc_auc"]*100  for r in results.values()]
x = np.arange(len(names))
b1 = axes[1].bar(x-0.175, accs, 0.35, label="Accuracy", color="#2563EB")
b2 = axes[1].bar(x+0.175, aucs, 0.35, label="ROC-AUC",  color="#DC2626")
axes[1].set_xticks(x); axes[1].set_xticklabels([n.replace(" ","\n") for n in names])
axes[1].set_ylabel("Score (%)"); axes[1].set_ylim(60,105)
axes[1].set_title("Model Comparison", fontweight="bold"); axes[1].legend()
for bar in list(b1)+list(b2):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
        f"{bar.get_height():.1f}%", ha="center", fontsize=9, fontweight="bold")
plt.tight_layout()
plt.savefig("model_comparison.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# ---- CELL 12: SHAP Explainability ----
xgb_model = results["XGBoost"]["model"]
explainer  = shap.TreeExplainer(xgb_model)
shap_vals  = explainer.shap_values(X_test)
plt.figure(figsize=(10,7))
shap.summary_plot(shap_vals, X_test, plot_type="bar", max_display=15, show=False)
plt.title("SHAP Feature Importance - Top 15 Churn Drivers", fontweight="bold")
plt.tight_layout(); plt.savefig("shap_importance.png", dpi=150, bbox_inches="tight"); plt.show()
plt.figure(figsize=(10,7))
shap.summary_plot(shap_vals, X_test, max_display=15, show=False)
plt.title("SHAP Beeswarm", fontweight="bold")
plt.tight_layout(); plt.savefig("shap_beeswarm.png", dpi=150, bbox_inches="tight"); plt.show()
shap_df = pd.DataFrame({"Feature": X.columns,
    "SHAP_Importance": np.abs(shap_vals).mean(axis=0)
}).sort_values("SHAP_Importance", ascending=False)
shap_df.to_csv("shap_importance.csv", index=False)
print("Saved: shap_importance.csv"); print(shap_df.head(10))

In [ ]:
# ---- CELL 13: Export for Power BI ----
joblib.dump(xgb_model, "churn_model.pkl")
df_export = df.copy()
X_full = df_model.drop("Churn", axis=1)
df_export["ChurnProbability"] = xgb_model.predict_proba(X_full)[:,1]
df_export["RiskSegment"] = pd.cut(df_export["ChurnProbability"],
    bins=[0,0.3,0.6,1.0], labels=["Low Risk","Medium Risk","High Risk"])
df_export.to_csv("churn_data_powerbi.csv", index=False)
print("Saved: churn_data_powerbi.csv  <-- import into Power BI")
print(df_export["RiskSegment"].value_counts())


In [ ]:
# ---- CELL 14: Business Insights ----
churn_rate   = (df["Churn"]=="Yes").mean()*100
revenue_risk = df[df["Churn"]=="Yes"]["MonthlyCharges"].sum()
high_risk    = (df_export["RiskSegment"]=="High Risk").sum()
mtm_churn    = df[df["Contract"]=="Month-to-month"]["Churn"].value_counts(normalize=True)["Yes"]*100
print("="*55)
print("        BUSINESS INSIGHTS SUMMARY")
print("="*55)
print(f"  Overall Churn Rate        : {churn_rate:.1f}%")
print(f"  Monthly Revenue at Risk   : ${revenue_risk:,.0f}")
print(f"  High Risk Customers       : {high_risk:,}")
print(f"  Month-to-Month Churn Rate : {mtm_churn:.1f}%")
print("="*55)
print("  KEY FINDINGS:")
print("  1. Month-to-month customers churn 3x more than annual")
print("  2. New customers (0-12 months) are highest risk")
print("  3. Electronic check = highest churn payment method")
print("  4. Tenure is #1 churn predictor (SHAP)")
print("  RECOMMENDATIONS:")
print("  1. Convert month-to-month to annual with discounts")
print("  2. Launch 12-month onboarding retention program")
print("  3. Proactive outreach to High Risk segment")
print("="*55)